# Настройка окружения для обучения LLM с ClearML

Этот ноутбук поможет вам:
- Настроить **ClearML** для отслеживания экспериментов
- Установить зависимости из репозитория
- Клонировать проект
- Авторизоваться в **Hugging Face Hub**
- Запустить обучение Qwen2.5-0.5B с разными оптимизаторами

> **Совет**: Убедитесь, что в Colab выбран **GPU** (Среда выполнения → Сменить тип среды выполнения → T4 GPU / V100).

# Монтируем Google Drive для сохранения результатов

In [1]:

from google.colab import drive
drive.mount('/content/drive')

NotImplementedError: Mounting drive is unsupported in this environment. Use PyDrive2 instead. See examples at https://colab.research.google.com/notebooks/io.ipynb#scrollTo=7taylj9wpsA2.

In [16]:
# Установка всех зависимостей из requirements.txt репозитория
# Используем %pip для корректной работы в Colab
# Предварительно установим git, если его нет (обычно уже есть)


# Клонируем репозиторий
!git clone https://github.com/Se1ecta/huawei-research.git /kaggle/working/huawei-research



# Устанавливаем зависимости из requirements.txt
#!pip install -r /content/huawei-research/requirements.txt

# Можем установить просто установить селдующие библиотеки, осталное уже есть в colab

%pip install clearml
%pip install transformers==4.51.3

Cloning into '/kaggle/working/huawei-research'...
remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 58 (delta 21), reused 53 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (58/58), 168.05 KiB | 18.67 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 🔑 Настройка ClearML

Вам понадобятся **API access key** и **secret key** из вашего аккаунта ClearML (можно получить на [app.clear.ml](https://app.clear.ml) → Settings → Workspace → Create new credentials).

**Вариант A (рекомендуемый)**: Используйте Colab Secrets для безопасного хранения ключей.
1. Откройте `🔑 Секреты` (панель слева в Colab)
2. Добавьте секреты: `CLEARML_API_ACCESS_KEY`, `CLEARML_API_SECRET_KEY`
3. Включите "Разрешить доступ через ноутбук"

**Вариант B**: Вставьте ключи прямо в код (не публикуйте ноутбук с ключами в открытом доступе!).


In [3]:
import os
from google.colab import userdata

# Пытаемся получить ключи из Colab Secrets
try:
    access_key = userdata.get('CLEARML_API_ACCESS_KEY')
    secret_key = userdata.get('CLEARML_API_SECRET_KEY')
    print("✅ Ключи ClearML загружены из Colab Secrets")
except Exception as e:
    print("⚠️ Не найдены секреты в Colab. Введите ключи вручную (небезопасно для шаринга).")
    access_key = input("Введите CLEARML_API_ACCESS_KEY: ")
    secret_key = input("Введите CLEARML_API_SECRET_KEY: ")

# Устанавливаем переменные окружения для ClearML
os.environ['CLEARML_WEB_HOST'] = 'https://app.clear.ml/'
os.environ['CLEARML_API_HOST'] = 'https://api.clear.ml'
os.environ['CLEARML_FILES_HOST'] = 'https://files.clear.ml'
os.environ['CLEARML_API_ACCESS_KEY'] = access_key
os.environ['CLEARML_API_SECRET_KEY'] = secret_key

print("✅ ClearML настроен. При запуске обучения эксперимент автоматически появится в веб-интерфейсе.")

⚠️ Не найдены секреты в Colab. Введите ключи вручную (небезопасно для шаринга).


Введите CLEARML_API_ACCESS_KEY:  NGTSC8Q7MH4V5O3GJ7YDCHM7L9J9OY
Введите CLEARML_API_SECRET_KEY:  kbRdFoxz6pfpcENaTtSWk9N5rQY0L4gxF4svHWFKGE8aeqHahnn8LHOo0_WZEEh7Nsg


✅ ClearML настроен. При запуске обучения эксперимент автоматически появится в веб-интерфейсе.


## Авторизация в Hugging Face Hub

## 🔑 Логин в Hugging Face

Чтобы пушить модели на Hub (опция `--push_to_hub=True`), нужен токен доступа. Получить его можно на [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (тип `write`).

In [4]:
from huggingface_hub import notebook_login

notebook_login()
# Появится виджет для ввода токена. Вставьте токен и нажмите "Login".

## Проверка GPU и дополнительных параметров

In [7]:
import torch

# Проверяем, доступен ли GPU
if torch.cuda.is_available():
    print(f"✅ GPU найден: {torch.cuda.get_device_name(0)}")
    print(f"   Количество памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU не обнаружен. Обучение будет очень медленным. Перезапустите среду выполнения и выберите GPU.")

# Проверяем, что репозиторий склонирован
!ls -la ./content/huawei-research/src/

✅ GPU найден: Tesla T4
   Количество памяти: 15.64 GB
total 76
drwxr-xr-x 4 root root  4096 Apr 20 07:18 .
drwxr-xr-x 7 root root  4096 Apr 20 07:18 ..
-rw-r--r-- 1 root root 41719 Apr 20 07:18 data_utils.py
-rw-r--r-- 1 root root     0 Apr 20 07:18 __init__.py
drwxr-xr-x 2 root root  4096 Apr 20 07:18 mezo
drwxr-xr-x 2 root root  4096 Apr 20 07:18 optimizers
-rw-r--r-- 1 root root  8551 Apr 20 07:18 train.py
-rw-r--r-- 1 root root   234 Apr 20 07:18 utils.py


## Запуск обучения (пример с AdamW)

## 🧪 Запуск эксперимента

Команда ниже запускает обучение модели **Qwen/Qwen2.5-0.5B** на 1 эпоху с оптимизатором AdamW.
Все метрики и логи автоматически отправятся в ClearML.

In [17]:
# Переходим в рабочую директорию (опционально)
import os
os.chdir('/kaggle/working/huawei-research')

# Можно изменить параметры: batch_size, learning_rate, оптимизатор и т.д.
!accelerate launch --num_processes 2 src/train.py \
  --model_name Qwen/Qwen2.5-0.5B \
  --optimizer adamw \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --learning_rate 5e-5 \
  --seq_length 512 \
  --num_train_epochs 1 \
  --lr_scheduler_type cosine \
  --warmup_ratio 0.01 \
  --weight_decay 0.05 \
  --logging_steps 10 \
  --push_to_hub True \
  --report_to clearml \
  --seed 42 \
  --output_dir ./Qwen2.5-0.5B_adamw

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-04-20 07:37:05.949421: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776670625.977752     944 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776670625.986318     944 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has al